In [1]:
import numpy as np
import torch
import scanpy as sc
import anndata as ad
import os
import pandas as pd
from utils.preprocess import *

In [2]:
import os
import sys
sys.path.append(os.path.abspath("conditional-flow-matching"))
    
import matplotlib.pyplot as plt
import numpy as np
import scanpy as sc
import torch
import torchsde
from torchdyn.core import NeuralODE
from tqdm import tqdm

from torchcfm.conditional_flow_matching import *
from torchcfm.models import MLP
from torchcfm.utils import plot_trajectories, torch_wrapper

[KeOps] Warning : 
    The default C++ compiler could not be found on your system.
    You need to either define the CXX environment variable or a symlink to the g++ command.
    For example if g++-8 is the command you can do
      import os
      os.environ['CXX'] = 'g++-8'
    
[KeOps] Warning : Cuda libraries were not detected on the system or could not be loaded ; using cpu only mode


In [3]:
import random
import umap
from models.modules import *

In [4]:
import matplotlib.pyplot as plt
%matplotlib inline

In [5]:
from scripts.run_model import *

In [6]:
%reload_ext autoreload
%autoreload 2

In [7]:
############################################################

In [8]:
config = Config(
    {        
        "max_epochs": 15000,
        "lr": 1e-4,
        "vae": True,
        "dropout": 0.0, #TODO: not used rn
        "pc_dim": 50,
        "hidden_dim": 512,
        "latent_dim": 100,
        "batch_size": 256,
        "num_freq": 16,
        "num_layers": 4,
        "sigma": 0.1,
        "constrain": True,
        "force_cpu": False,
        "gradient_clip_val": 0,
        "accumulate_grad_batches": 10,
    }
)

project = "cvae-ipynb"

In [9]:
wandb.finish()

In [10]:
pl_model, adata, values, conditions, dataset, t_min, t_max = run_model(config, project, return_data = True)
model = pl_model

wandb: Currently logged in as: az831 (az831-new-york-genome-center) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.


/home/azweig/miniconda3/envs/spatialenv/lib/python3.10/site-packages/scanpy/preprocessing/_pca/__init__.py:379: ImplicitModificationWarning: Setting element `.obsm['X_pca']` of view, initializing view as actual.
  adata.obsm[key_obsm] = X_pca
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/home/azweig/miniconda3/envs/spatialenv/lib/python3.10/site-packages/pytorch_lightning/loggers/wandb.py:397: There is a wandb run already in progress and newly created instances of `WandbLogger` will reuse this run. If this is not desired, call `wandb.finish()` before instantiating `WandbLogger`.
wandb: logging graph, to disable use `wandb.watch(log_graph=False)`
You are using a CUDA device ('NVIDIA GeForce RTX 4080 SUPER') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/do

epoch,▁▁▁▁▁▁▁▂▂▂▂▂▂▃▃▄▄▄▄▄▄▄▄▄▅▅▅▅▅▆▆▆▇▇▇▇▇▇▇█
train_loss,▁▆▃▆▅▇▇▁▆▂▂▆▂▃▂▇▃▇▅▅▆▇▁▃█▂▇▁▄▇▆▆▇▂▁▆▁▇▇▃
trainer/global_step,▁▁▁▁▁▁▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▄▄▄▄▅▅▅▆▆▆▆▆▆▇▇▇▇██
epoch,14999
train_loss,4.71762
trainer/global_step,14999


AttributeError: 'ConditionalVAE' object has no attribute 'flow_net'

In [12]:
def save_umap(value):
    adata_local = adata[(adata.obs['gene_target'] == value) | (adata.obs['gene_target'] == 'ctrl-inj')]
    
    X = adata_local.obsm['X_pca']
    umap_model = umap.UMAP().fit(X) #pca of ctrl adata
    
    adata_local.obsm["X_umap"] = umap_model.embedding_
    return adata_local, umap_model

In [13]:
def plot_trajectories(model, value, num_traj, t_min, t_max):
    adata_local, umap_model = save_umap(value)

    adata_ctrl = adata_local[adata_local.obs['gene_target'] == 'ctrl-inj']
    adata_per = adata_local[adata_local.obs['gene_target'] == value]

    timepoints = sorted(adata_per.obs['timepoint'].unique().tolist())
    t0, t1 = min(timepoints), max(timepoints)

    print(t0, t1)
    
    c = conditions[value]
    c = c.unsqueeze(0).repeat(num_traj, 1)
    
    X = adata_per.obsm['X_pca']
    X0 = adata_per[adata_per.obs['timepoint'] == min(timepoints)].obsm['X_pca']

    device = next(model.parameters()).device
    
    node = NeuralODE(WrappedVectorField(model, c), solver="dopri5", sensitivity="adjoint")
    with torch.no_grad():
        traj = node.trajectory(
            torch.from_numpy(X0[:num_traj]).float().to(device),
            t_span=torch.linspace(t0, t1, 400),
        ).cpu()

    trajectories = [traj[:,i].numpy() for i in range(traj.shape[1])]
    trajectories_umap = [umap_model.transform(traj) for traj in trajectories]

    fig, (ax0, ax1, ax2, ax3) = plt.subplots(4, figsize=(16, 16))

    sc.pl.umap(
        adata_local,
        color="tissue",
        # Setting a smaller point size to get prevent overlap
        size=2,
        ax=ax0,
        show=False
    )
    
    sc.pl.umap(
        adata_local,
        color="gene_target",
        # Setting a smaller point size to get prevent overlap
        size=2,
        ax=ax1,
        show=False
    )

    sc.pl.umap(adata_per, color="timepoint", ax=ax2, show=False)

    
    sc.pl.umap(adata_per, color="timepoint", ax=ax3, show=False)
    
    for traj in trajectories_umap:
        ax3.plot(traj[:, 0], traj[:, 1], marker=".", linestyle="-", alpha=0.7, lw=1.5)
    
    plt.show()

    return trajectories

In [14]:
#TODO: rewrite for conditional model as above
def plot_vector_field(model, value, num_traj, t_min, t_max):
    adata_local, umap_model = save_umap(value)

    adata_ctrl = adata_local[adata_local.obs['gene_target'] == 'ctrl-inj']
    adata_per = adata_local[adata_local.obs['gene_target'] == value]

    timepoints = sorted(adata_local.obs['timepoint'].unique().tolist())
    t0, t1 = min(timepoints), max(timepoints)
    
    c = conditions[value]
    c = c.unsqueeze(0).repeat(num_traj, 1)
    
    X = adata_per.obsm['X_pca']
    X0 = adata_per[adata_per.obs['timepoint'] == min(timepoints)].obsm['X_pca']

    device = next(model.parameters()).device
    
    node = NeuralODE(WrappedVectorField(model, c), solver="dopri5", sensitivity="adjoint")
    with torch.no_grad():
        traj = node.trajectory(
            torch.from_numpy(X0[:num_traj]).float().to(device),
            t_span=torch.linspace(t0, t1, 10),
        ).cpu()
    
    trajectories = [traj[:,i].numpy() for i in range(traj.shape[1])]
    trajectories_umap = [umap_model.transform(traj) for traj in trajectories]

    fig, (ax1, ax2) = plt.subplots(2, figsize=(16, 16))
    
    sc.pl.umap(
        adata_per,
        color="tissue",
        # Setting a smaller point size to get prevent overlap
        size=2,
        ax=ax1,
        show=False
    )
    
    sc.pl.umap(adata_per, color="timepoint", ax=ax2, show=False)
    
    for traj in trajectories_umap:
        print(traj[0], traj[1])
        diffs = np.diff(traj, axis=0)
        ax2.quiver(traj[0, 0], traj[0, 1], diffs[0, 0], diffs[0, 1],
                   angles='xy', scale_units='xy', color='r', 
                   linewidth=0.2, alpha=0.5)
    
    plt.show()

In [15]:
# value = 'tbx16'
# # value = 'ctrl-inj'
# print(value)
# trajectories = plot_trajectories(model, value, 10, t_min, t_max)

In [16]:
# value = 'ctrl-inj'
# plot_vector_field(model, value, 50, t_min, t_max)

In [17]:
# for traj in trajectories:
#     steps = np.diff(traj, axis=0)
#     steps = np.linalg.norm(steps, axis=1)
#     print("_")
#     print(np.median(steps))
#     print(np.max(steps))

In [18]:
import ot
def ot_dist(X, Y, ot_epsilon = 0.1):
    a = np.ones(X.shape[0]) / X.shape[0]
    b = np.ones(Y.shape[0]) / Y.shape[0]
    M = ot.dist(X, Y)
    ot_epsilon = 0.1
    dist = ot.sinkhorn2(a, b, M, ot_epsilon, method='sinkhorn_log')
    return dist

In [25]:
#TODO: code for removing the test timepoint?
def predict(model, value, num_traj, t_min, t_max, t):
    
    adata_local, umap_model = save_umap(value)
    
    adata_ctrl = adata_local[adata_local.obs['gene_target'] == 'ctrl-inj']
    adata_per = adata_local[adata_local.obs['gene_target'] == value]  

    X = adata_per[adata_per.obs['timepoint'] == t].obsm['X_pca']

    device = next(model.parameters()).device
    
    c = conditions[value]
    c = c.unsqueeze(0).repeat(num_traj, 1).to(device)
    t = torch.tensor(t).unsqueeze(0).repeat(num_traj).to(device)
    
    samples = model.sample(c, t, num_traj).detach().cpu().numpy()
    print(X.shape)
    print(samples.shape)
        
    emp_dist = ot_dist(samples, np.array(X))

    ctrl_dist=None
    # Y = adata_ctrl[adata_ctrl.obs['timepoint'] == timepoints[index]].obsm['X_pca']
    # indices = np.random.choice(Y.shape[0], num_traj, replace=False)
    # Y = Y[indices]
    # ctrl_dist = ot_dist(Y, X1)

    return emp_dist, ctrl_dist

In [26]:
value = 'tbx16'
t = 24
num_traj = 2000

print(predict(model, value, num_traj, t_min, t_max, t))

/tmp/ipykernel_3253426/2814889023.py:7: ImplicitModificationWarning: Setting element `.obsm['X_umap']` of view, initializing view as actual.
  adata_local.obsm["X_umap"] = umap_model.embedding_


(19389, 50)
(2000, 50)
(np.float64(116.35432331292162), None)


In [ ]:
#Some kind of result???
#On the all data setting, with t in [18, 24, 36]
#Without any kind of masking of course
#Without CFG
#on 'tbx16'
#result of ot.sinkhorn2(ot_epsilon=0.1, method='sinkhorn_log')
#(rollout from 18->24 of learned model, control at 24)
#with 2000 trajectories / random control samples
#(np.float64(75.01488016079345), np.float64(84.0595873155962))

#for CVAE
#(np.float64(116.35432331292162))
#clearly need to regularize...

In [ ]:
#######################################